# Collection Lineage → UMM-C MetadataAssociations

Implements **Collection_Lineage_Generation_Skill** for a single CMR collection.

**Input:** a collection `SHORT_NAME` and `VERSION`
**Output:** the collection's full UMM-C record (v1.18.6) with a validated `MetadataAssociations` array, plus an evidence file listing the source quote behind each association.

## Workflow
| Step | What happens | Skill step |
|---|---|---|
| 0 | Fetch the target collection's UMM-C record from CMR | Execution template: `get_collection_metadata` |
| 1 | Find the ATBD / User Guide in `RelatedUrls`, download and extract text | `fetch_atbd_from_metadata` |
| 2 | Build a catalog of candidate CMR collections (same instrument) | — |
| 3 | Extract lineage from the documents (Claude, or a rule-based fallback) and verify every quote against the document text | `extract_lineage_from_atbd` |
| 4 | Map relationships to UMM-C types, resolve versions in CMR, write descriptions | Steps 1–2 |
| 5 | Validate each association (skill rules) and the whole record (official UMM-C JSON schema) | Step 3 |
| 6 | Merge into the UMM-C record and save | Steps 4–5 |

## Lineage extraction modes
* **Claude (recommended):** documents often describe inputs generically ("the Level 1A granules", "the geolocation file") without naming a ShortName. Claude maps those descriptions onto the CMR candidate catalog and returns a verbatim evidence quote for each relationship. Needs Anthropic credentials (`ANTHROPIC_API_KEY`, or an `ant auth login` profile).
* **Rule-based fallback:** finds ShortNames that are written in the documents and classifies them with keyword rules. It needs no credentials but misses inputs that are described only generically.

Every relationship's evidence quote is checked against the extracted document text. By default, relationships whose quote can't be found are dropped.

## Requirements
`pip install requests pypdf jsonschema referencing anthropic`

In [ ]:
from pathlib import Path

# ---- Inputs ---------------------------------------------------------------
SHORT_NAME = "MOD02QKM"
VERSION = "7"
PROVIDER = None            # e.g. "LAADS"; only needed if ShortName+Version matches more than one record

# ---- Options --------------------------------------------------------------
USE_LLM = "auto"           # True | False | "auto" (use Claude when Anthropic credentials are found)
CLAUDE_MODEL = "claude-opus-5-5"
CLAUDE_EFFORT = "high"     # low | medium | high | xhigh | max
REUSE_CLAUDE_RESPONSE = True       # reuse a saved Claude response when the prompt is unchanged
MAX_DOCUMENTS = 4          # documentation files to read, in priority order (ATBD first)
MERGE_MODE = "merge"       # "merge": keep existing associations for EntryIds not regenerated; "replace": overwrite
REQUIRE_VERIFIED_EVIDENCE = True   # drop relationships whose quote is not found in the document text
KEEP_ENTRIES_NOT_IN_CMR = False    # keep associations to ShortNames with no CMR record at all (Version omitted)

CMR_BASE_URL = "https://cmr.earthdata.nasa.gov"
UMM_C_VERSION = "1.18.6"
WORK_DIR = Path("lineage_output")

In [ ]:
import hashlib
import json
import os
import re
from datetime import datetime, timezone
from html.parser import HTMLParser

import requests
from pypdf import PdfReader

# UMM-C v1.18.6 MetadataAssociateTypeEnum
VALID_ASSOCIATION_TYPES = [
    "SCIENCE ASSOCIATED", "DEPENDENT", "INPUT",
    "PARENT", "CHILD", "RELATED", "LARGER CITATION WORKS",
]

# Skill Step 1d mapping (skill v2.1). Each relationship describes the *related* collection X relative to the target T.
LINEAGE_TO_UMM_TYPE = {
    "derived_from": "PARENT",
    "aggregated_from": "PARENT",
    "input_to": "INPUT",
    "component_of": "INPUT",
    "template_of": "RELATED",
    "version_of": "RELATED",
    "equivalent_product": "RELATED",
    "parallel_product": "RELATED",
    "related_to": "RELATED",
    "downstream_product": "CHILD",
}
LINEAGE_RELATIONSHIPS = list(LINEAGE_TO_UMM_TYPE)

session = requests.Session()
session.headers["User-Agent"] = "collection-lineage-umm-c-export-notebook"
DOC_DIR = WORK_DIR / "docs" / f"{SHORT_NAME}_{VERSION}"
DOC_DIR.mkdir(parents=True, exist_ok=True)

## Step 0 — Fetch the target collection from CMR
The record is requested in UMM-C 1.18.6 so the output conforms to the schema version the skill targets.

In [ ]:
def cmr_get(path, params, accept=None):
    headers = {"Accept": accept} if accept else {}
    r = session.get(f"{CMR_BASE_URL}{path}", params=params, headers=headers, timeout=60)
    r.raise_for_status()
    return r


def fetch_target_umm(short_name, version, provider=None):
    params = {"short_name": short_name, "version": version, "page_size": 10}
    if provider:
        params["provider"] = provider
    r = cmr_get("/search/collections.umm_json", params,
                accept=f"application/vnd.nasa.cmr.umm_results+json; version={UMM_C_VERSION}")
    items = r.json()["items"]
    if not items:
        raise ValueError(f"No CMR collection found for {short_name} version {version}")
    if len(items) > 1:
        found = [f"{i['meta']['concept-id']} ({i['meta']['provider-id']})" for i in items]
        raise ValueError(f"{short_name} v{version} matches {len(items)} records: {found}. Set PROVIDER.")
    return items[0]["meta"], items[0]["umm"]


_versions_cache = {}

def cmr_versions(short_name):
    # All versions of a ShortName that exist in CMR (any provider)
    if short_name not in _versions_cache:
        r = cmr_get("/search/collections.json", {"short_name": short_name, "page_size": 100})
        _versions_cache[short_name] = sorted({e["version_id"] for e in r.json()["feed"]["entry"]})
    return _versions_cache[short_name]


target_meta, target_umm = fetch_target_umm(SHORT_NAME, VERSION, PROVIDER)
target_instruments = sorted({i["ShortName"] for p in target_umm.get("Platforms", []) for i in p.get("Instruments", [])})
target_platforms = [p["ShortName"] for p in target_umm.get("Platforms", [])]

print(f"Concept ID : {target_meta['concept-id']} (revision {target_meta['revision-id']})")
print(f"Provider   : {target_meta['provider-id']}   native-id: {target_meta['native-id']}")
print(f"Title      : {target_umm['EntryTitle']}")
print(f"Level      : {target_umm.get('ProcessingLevel', {}).get('Id')}")
print(f"Platforms  : {target_platforms}   Instruments: {target_instruments}")
print(f"Existing MetadataAssociations: {len(target_umm.get('MetadataAssociations') or [])}")

## Step 1 — Find and read the documentation
Documentation links are ranked by `RelatedUrls.Subtype`, with the ATBD first. PDFs are parsed with `pypdf`; HTML is reduced to plain text. Page markers (`[Page n]`) are kept so evidence can be located.

In [ ]:
DOC_SUBTYPE_PRIORITY = {
    "ALGORITHM THEORETICAL BASIS DOCUMENT (ATBD)": 1,
    "USER'S GUIDE": 2,
    "DATA PRODUCT SPECIFICATION": 3,
    "SCIENCE DATA PRODUCT SPECIFICATION (SDPS)": 3,
    "PRODUCT HISTORY": 3,
    "PROCESSING HISTORY": 3,
    "PRODUCT QUALITY ASSESSMENT": 4,
    "GENERAL DOCUMENTATION": 4,
    "PI DOCUMENTATION": 4,
    "READ-ME": 4,
}


def rank_documents(related_urls):
    docs, seen = [], set()
    for u in related_urls or []:
        url = u.get("URL", "")
        subtype = (u.get("Subtype") or "").upper()
        priority = DOC_SUBTYPE_PRIORITY.get(subtype)
        if priority is None and "ATBD" in url.upper():
            priority = 1
        if priority is None or url in seen:
            continue
        seen.add(url)
        docs.append({"url": url, "subtype": subtype or "ATBD (from URL)", "priority": priority,
                     "description": u.get("Description", "")})
    return sorted(docs, key=lambda d: d["priority"])


class _HTMLText(HTMLParser):
    def __init__(self):
        super().__init__()
        self.parts, self._skip = [], 0
    def handle_starttag(self, tag, attrs):
        self._skip += tag in ("script", "style")
    def handle_endtag(self, tag):
        self._skip -= tag in ("script", "style")
    def handle_data(self, data):
        if not self._skip:
            self.parts.append(data)


def download(url):
    name = re.sub(r"[^A-Za-z0-9._-]", "_", url.rstrip("/").rsplit("/", 1)[-1])[:120] or "document"
    path = DOC_DIR / name
    if not path.exists():
        r = session.get(url, timeout=180, allow_redirects=True)
        r.raise_for_status()
        if "urs.earthdata.nasa.gov" in r.url:
            raise RuntimeError("requires Earthdata Login")
        path.write_bytes(r.content)
    return path


def extract_text(path):
    data = path.read_bytes()
    if data[:5] == b"%PDF-":
        reader = PdfReader(path)
        return "\n".join(f"[Page {i}]\n{p.extract_text() or ''}" for i, p in enumerate(reader.pages, 1))
    parser = _HTMLText()
    parser.feed(data.decode("utf-8", errors="replace"))
    return re.sub(r"\n\s*\n+", "\n\n", "".join(parser.parts))


def _norm(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()


documents = []
for d in rank_documents(target_umm.get("RelatedUrls")):
    if len(documents) >= MAX_DOCUMENTS:
        break
    try:
        text = extract_text(download(d["url"]))
    except Exception as e:
        print(f"  skipped {d['url']}: {e}")
        continue
    if len(text.strip()) < 500:
        print(f"  skipped {d['url']}: too little extractable text")
        continue
    doc_id = f"DOC{len(documents) + 1}"
    label = "ATBD" if d["priority"] == 1 else (d["description"] or d["subtype"].title())
    documents.append({**d, "id": doc_id, "label": label, "text": text, "norm": _norm(text)})
    print(f"{doc_id}: [{d['subtype']}] {label} — {len(text):,} chars\n      {d['url']}")

if not documents:
    print("No readable documentation found; only platform-counterpart relationships can be inferred.")

## Step 2 — Candidate collection catalog
CMR collections that share the target's instrument(s), or its platform(s) when no instrument is listed. Claude chooses ShortNames from this list, and the rule-based extractor uses it to recognize ShortNames in the text.

In [ ]:
def build_candidate_catalog():
    if target_instruments:
        facet = [("instrument", i) for i in target_instruments]
    elif target_platforms:
        facet = [("platform", p) for p in target_platforms]
    else:
        return {}
    params = facet + [("page_size", 2000), ("options[instrument][ignore_case]", "true"),
                      ("options[platform][ignore_case]", "true")]
    r = cmr_get("/search/collections.json", params)
    if int(r.headers.get("CMR-Hits", 0)) > 2000 and target_platforms:
        # Too broad (e.g. a multi-mission instrument): narrow to the target's platforms
        r = cmr_get("/search/collections.json", params + [("platform", p) for p in target_platforms])
    catalog = {}
    for e in r.json()["feed"]["entry"]:
        c = catalog.setdefault(e["short_name"], {"versions": set(), "title": e["title"],
                                                 "level": e.get("processing_level_id", "")})
        c["versions"].add(e["version_id"])
    for sn, c in catalog.items():
        _versions_cache.setdefault(sn, sorted(c["versions"]))
    return catalog


catalog = build_candidate_catalog()
catalog.pop(SHORT_NAME, None)
catalog_text = "\n".join(
    f"{sn} | versions: {', '.join(sorted(c['versions']))} | level: {c['level']} | {c['title']}"
    for sn, c in sorted(catalog.items()))
print(f"{len(catalog)} candidate collections (~{len(catalog_text) // 4:,} tokens)")

## Step 3 — Extract lineage from the documents
Both extractors return the same record shape:
`entry_id`, `relationship` (a key of `LINEAGE_TO_UMM_TYPE`), `document_id`, `section`, `evidence_quote`, `detail`, `explanation`.

### 3a. Rule-based extractor (fallback)

In [ ]:
# Platform-prefix counterparts (Terra/Aqua MODIS, SNPP/NOAA-20/NOAA-21 VIIRS)
COUNTERPART_PREFIXES = [("MOD", "MYD"), ("MYD", "MOD"), ("VNP", "VJ1"), ("VJ1", "VNP")]
RULE_STRENGTH = {"derived_from": 4, "input_to": 3, "equivalent_product": 2, "related_to": 1}
DERIVED_RE = re.compile(r"\b(derived|generated|produced|computed|calculated|aggregated|averaged|composited|created)\s+(from|using)\b", re.I)
INPUT_RE = re.compile(r"\binputs?\b|\bingest", re.I)
RULE_DETAIL = {
    "derived_from": "Derived from this collection (keyword match; review)",
    "input_to": "Named in an input statement (keyword match; review)",
    "equivalent_product": "Same product from the counterpart platform",
    "related_to": "Named alongside the target in the documentation (review)",
}


def platform_counterparts(short_name):
    return [new + short_name[len(old):] for old, new in COUNTERPART_PREFIXES if short_name.startswith(old)]


def extract_lineage_rules(documents):
    counterparts = set(platform_counterparts(SHORT_NAME))
    found = {}
    for doc in documents:
        page = None
        for chunk in re.split(r"(\[Page \d+\])", doc["text"]):
            if m := re.fullmatch(r"\[Page (\d+)\]", chunk):
                page = m.group(1)
                continue
            for sentence in re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", chunk)):
                for sn in {t.strip(".-") for t in re.findall(r"[A-Za-z0-9_.\-]+", sentence)} & catalog.keys():
                    m = DERIVED_RE.search(sentence)
                    if m and sentence.find(sn) > m.start():
                        rel = "derived_from"
                    elif INPUT_RE.search(sentence):
                        rel = "input_to"
                    elif sn in counterparts:
                        rel = "equivalent_product"
                    else:
                        rel = "related_to"
                    if RULE_STRENGTH[rel] > RULE_STRENGTH.get(found.get(sn, {}).get("relationship"), 0):
                        found[sn] = {"entry_id": sn, "relationship": rel, "document_id": doc["id"],
                                     "section": f"Page {page}" if page else "",
                                     "evidence_quote": sentence.strip()[:400], "detail": RULE_DETAIL[rel],
                                     "explanation": "Rule-based: ShortName mentioned in documentation."}
    # Drop other-platform counterparts of sibling products (e.g. MYD02HKM when the target is MOD02QKM)
    own_prefixes = [old for old, _ in COUNTERPART_PREFIXES if SHORT_NAME.startswith(old)]
    for sn in list(found):
        if sn in counterparts or any(sn.startswith(p) for p in own_prefixes):
            continue
        if any(c in found for c in platform_counterparts(sn)):
            del found[sn]
    for sn in counterparts:
        if sn not in found and cmr_versions(sn):
            found[sn] = {"entry_id": sn, "relationship": "equivalent_product", "document_id": "",
                         "section": "", "evidence_quote": "",
                         "detail": f"Same product from the counterpart platform ({sn[:3]} vs {SHORT_NAME[:3]})",
                         "explanation": "Rule-based: platform-prefix counterpart that exists in CMR."}
    return list(found.values())

### 3b. Claude extractor
One request containing the documents, the target summary and the candidate catalog. The output is constrained to a JSON schema (structured outputs). It uses adaptive thinking and server-side refusal fallback (`fallbacks: "default"`).

In [ ]:
LLM_OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "relationships": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "entry_id": {"type": "string"},
                    "relationship": {"type": "string", "enum": LINEAGE_RELATIONSHIPS},
                    "document_id": {"type": "string"},
                    "section": {"type": "string"},
                    "evidence_quote": {"type": "string"},
                    "detail": {"type": "string"},
                    "explanation": {"type": "string"},
                },
                "required": ["entry_id", "relationship", "document_id", "section",
                             "evidence_quote", "detail", "explanation"],
                "additionalProperties": False,
            },
        },
        "notes": {"type": "string"},
    },
    "required": ["relationships", "notes"],
    "additionalProperties": False,
}

SYSTEM_PROMPT = '''You are a NASA Earth science data provenance analyst. You read algorithm and product
documentation for a CMR collection (the target, T) and identify the other CMR collections it has a
documented lineage relationship with. The results become UMM-C MetadataAssociations, so precision matters
more than recall: report a relationship only when the documents support it.'''

TASK_PROMPT = '''Identify every CMR collection X that has a documented lineage relationship with the target T.

Classify each X using the definitions and decision procedure from the skill (Step 1a-1c).

Relationship values describe X relative to T:
- derived_from: T is produced directly from X's data (X is T's upstream/parent product, e.g. Level 1A -> Level 1B, daily -> 8-day).
- aggregated_from: T temporally or spatially aggregates/composites X.
- input_to: X is consumed as an input or ancillary input by T's algorithm (e.g. geolocation, cloud mask), without T being a reprocessing of X.
- component_of: X is a component or subset that T is assembled from.
- downstream_product: X is produced from T.
- parallel_product: X is produced by the same processing run or algorithm alongside T (a sibling product).
- equivalent_product: X is the same product from another platform or instrument.
- template_of / version_of / related_to: other documented relationships. Do not list other versions of T itself.

Decision procedure - apply in order, stop at the first yes:
1. X is produced from T -> downstream_product
2. T is produced directly from X's data -> aggregated_from (temporal/spatial aggregation) or derived_from (any other reprocessing)
3. T's algorithm consumes X without reprocessing X's data -> component_of (one of several assembled components) or input_to
4. X is produced by the same run as T -> parallel_product
5. X is the same product from another platform/instrument -> equivalent_product
6. Otherwise, a documented relationship -> version_of, template_of or related_to
PARENT vs INPUT test: if T is X's data carried forward (calibrated, retrieved, averaged), it is derived_from/aggregated_from; if X only supports T's processing (geolocation, masks, surface type), it is input_to. If X does both, choose derived_from.

Rules:
0. Direct relationships only: list collections that T's own processing uses or produces. Inputs of T's parent are not inputs of T; use related_to only if the documents make that relationship worth recording.
1. entry_id must be a ShortName exactly as written in the candidate catalog. Only if the documents name a collection that is not in the catalog may you use the ShortName as written in the documents.
2. Documents often describe an input generically ("the Level 1A granules", "the geolocation file"). Map such a description to a catalog ShortName when the instrument, platform and processing level make the mapping unambiguous, and say in the explanation that the mapping was inferred.
3. Skip things that are not distributed as CMR collections (look-up tables, software, ephemeris/attitude files) unless the catalog lists them.
4. evidence_quote must be a contiguous passage copied verbatim from one document (at most about 300 characters), and document_id must name that document. Quotes are machine-checked against the document text.
5. section: the section number and/or title where the quote appears, otherwise the [Page n] marker.
6. detail: a short phrase giving the processing method (derived/aggregated), X's role (input), or the nature of the relationship (other types).
7. Report each entry_id at most once, choosing the earliest match in the decision procedure.
8. Precision over recall: when the evidence is ambiguous, use related_to or omit X.'''


def llm_credentials_available():
    return bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN")
                or Path.home().joinpath(".config", "anthropic").exists())


def extract_lineage_llm(documents):
    import anthropic

    target_summary = json.dumps({
        "ShortName": SHORT_NAME, "Version": VERSION, "EntryTitle": target_umm["EntryTitle"],
        "ProcessingLevel": target_umm.get("ProcessingLevel", {}).get("Id"),
        "Platforms": target_platforms, "Instruments": target_instruments,
        "Abstract": target_umm.get("Abstract", ""),
    }, indent=1)
    content = [{"type": "text", "text": f'<document id="{d["id"]}" title="{d["label"]}" url="{d["url"]}">\n{d["text"]}\n</document>'}
               for d in documents]
    content.append({"type": "text", "text": (
        f"<target_collection>\n{target_summary}\n</target_collection>\n\n"
        f"<candidate_catalog>\nShortName | versions | level | title\n{catalog_text}\n</candidate_catalog>\n\n{TASK_PROMPT}")})

    # Saved responses are keyed by a hash of the full request, so a re-run with unchanged inputs costs nothing
    request_key = hashlib.sha256(json.dumps([CLAUDE_MODEL, CLAUDE_EFFORT, SYSTEM_PROMPT, content,
                                             LLM_OUTPUT_SCHEMA]).encode()).hexdigest()[:12]
    response_path = WORK_DIR / f"{SHORT_NAME}_V{VERSION}_claude_response_{request_key}.json"
    if REUSE_CLAUDE_RESPONSE and response_path.exists():
        print(f"Reusing saved Claude response {response_path}")
        return _parse_llm_response(json.loads(response_path.read_text()))

    client = anthropic.Anthropic()

    def call(use_fallbacks):
        extra = {"betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"} if use_fallbacks else {}
        with client.beta.messages.stream(
            model=CLAUDE_MODEL,
            max_tokens=64000,
            thinking={"type": "adaptive"},
            output_config={"effort": CLAUDE_EFFORT,
                           "format": {"type": "json_schema", "schema": LLM_OUTPUT_SCHEMA}},
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": content}],
            **extra,
        ) as stream:
            return stream.get_final_message()

    try:
        message = call(use_fallbacks=True)
    except anthropic.APIStatusError as e:
        # Some gateways and platforms reject server-side fallbacks; retry once without them
        if e.status_code >= 500 or isinstance(e, (anthropic.AuthenticationError, anthropic.RateLimitError)):
            raise
        print(f"Request with server-side fallbacks rejected ({e.message}); retrying without them.")
        message = call(use_fallbacks=False)

    if message.stop_reason == "refusal":
        raise RuntimeError(f"Claude declined the request: {message.stop_details}")
    if message.stop_reason == "max_tokens":
        raise RuntimeError("Claude's response hit max_tokens before completing")
    print(f"Claude: {message.usage.input_tokens:,} input / {message.usage.output_tokens:,} output tokens")
    response = message.to_dict()
    response_path.parent.mkdir(parents=True, exist_ok=True)
    response_path.write_text(json.dumps(response, indent=1, default=str))
    return _parse_llm_response(response)


def _extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Endpoints that don't enforce output_config.format return prose with an embedded JSON block
    print("Response was not pure JSON (structured output not enforced); extracting the embedded JSON.")
    if m := re.search(r"```(?:json)?\s*(.*?)```", text, re.S):
        return json.loads(m.group(1))
    start = min(i for i in (text.find("{"), text.find("[")) if i >= 0)
    return json.JSONDecoder().raw_decode(text[start:])[0]


def _parse_llm_response(response):
    # The JSON may be split across several text blocks; thinking blocks are skipped
    text = "".join(b.get("text", "") for b in response["content"] if b["type"] == "text").strip()
    result = _extract_json(text)
    if isinstance(result, list):
        result = {"relationships": result, "notes": ""}
    if result.get("notes"):
        print(f"Claude notes: {result['notes']}")
    relationships = []
    for rel in result.get("relationships", []):
        rel = {key: str(rel.get(key) or "") for key in LLM_OUTPUT_SCHEMA["properties"]["relationships"]["items"]["required"]}
        if rel["relationship"] not in LINEAGE_TO_UMM_TYPE:
            print(f"  {rel['entry_id']}: unknown relationship '{rel['relationship']}' treated as related_to")
            rel["relationship"] = "related_to"
        if rel["entry_id"]:
            relationships.append(rel)
    return relationships

### 3c. Run extraction and verify evidence
A quote counts as verified when it appears in the cited document after case and punctuation are normalized. To tolerate PDF extraction artifacts, a quote also passes when at least 80% of its 6-word shingles are present.

In [ ]:
def verify_quote(quote, doc_id):
    q = _norm(quote)
    if not q:
        return False
    # Check the cited document first, then the others (models occasionally cite the wrong document id)
    for doc in sorted(documents, key=lambda d: d["id"] != doc_id):
        if q in doc["norm"]:
            return doc["id"]
        words = q.split()
        if len(words) >= 8:
            shingles = [" ".join(words[i:i + 6]) for i in range(len(words) - 5)]
            if sum(s in doc["norm"] for s in shingles) / len(shingles) >= 0.8:
                return doc["id"]
    return False


use_llm = llm_credentials_available() if USE_LLM == "auto" else bool(USE_LLM)
extraction_method = "rules"
if use_llm and documents:
    try:
        relationships = extract_lineage_llm(documents)
        extraction_method = f"claude ({CLAUDE_MODEL})"
    except Exception as e:
        print(f"Claude extraction failed ({type(e).__name__}: {e}); falling back to rule-based extraction.")
        relationships = extract_lineage_rules(documents)
else:
    if USE_LLM == "auto":
        print("No Anthropic credentials found; using rule-based extraction.")
    relationships = extract_lineage_rules(documents)

for rel in relationships:
    rel["entry_id"] = rel["entry_id"].strip()
    if not rel["evidence_quote"]:
        rel["verified"] = None          # heuristic relationship with no quote (e.g. platform counterpart)
    else:
        verified_in = verify_quote(rel["evidence_quote"], rel["document_id"])
        rel["verified"] = bool(verified_in)
        if verified_in:
            rel["document_id"] = verified_in

relationships = [r for r in relationships if r["entry_id"] != SHORT_NAME]
print(f"\nExtraction method: {extraction_method}")
print(f"{'EntryId':<22}{'relationship':<20}{'evidence':<11}section")
for r in relationships:
    status = {True: "verified", False: "NOT FOUND", None: "heuristic"}[r["verified"]]
    print(f"{r['entry_id']:<22}{r['relationship']:<20}{status:<11}{r['document_id']} {r['section']}")

## Step 4 — Build MetadataAssociations (skill Steps 1–2)
* **Type**: `map_to_umm_c_association_type` (skill Step 1d).
* **Version**: the target's version if the related collection has it in CMR. Otherwise the latest non-NRT version, flagged in the Description. If CMR has only NRT records, or no record, Version is omitted.
* **Description**: built in the skill's `generate_description` format and capped at 4000 characters.

In [ ]:
def map_to_umm_c_association_type(lineage_relationship):
    return LINEAGE_TO_UMM_TYPE.get(lineage_relationship.lower(), "RELATED")


def _version_key(v):
    return tuple(int(n) for n in re.findall(r"\d+", v)) or (0,)


def resolve_version(short_name):
    versions = cmr_versions(short_name)
    if not versions:
        return None, f"No {short_name} collection was found in CMR; Version omitted."
    if VERSION in versions:
        return VERSION, ""
    standard = [v for v in versions if "NRT" not in v.upper()]
    if standard:
        best = max(standard, key=_version_key)
        return best, f"Version {VERSION} of {short_name} is not in CMR; latest available version {best} used."
    return None, f"Only {', '.join(versions)} of {short_name} exist in CMR (no standard version); Version omitted."


def generate_description(related_sn, umm_type, rel):
    detail = (rel.get("detail") or "").strip().rstrip(".")
    section = re.sub(r"^section\s+", "", (rel.get("section") or "").strip(), flags=re.I)
    if umm_type == "PARENT":
        desc = f"{SHORT_NAME} is derived from {related_sn}. Processing: {detail or 'Not specified'}. "
    elif umm_type == "INPUT":
        desc = f"{related_sn} serves as input to {SHORT_NAME}. Role: {detail or 'Primary input'}. "
    elif umm_type == "CHILD":
        desc = f"{related_sn} is derived from {SHORT_NAME}. Processing: {detail or 'Not specified'}. "
    elif umm_type == "RELATED":
        desc = f"{related_sn} is related to {SHORT_NAME}. Relationship: {detail or 'Associated collection'}. "
    else:
        desc = f"{SHORT_NAME} uses {related_sn}. "
    doc = next((d for d in documents if d["id"] == rel.get("document_id")), None)
    if doc and section:
        desc += f"{doc['label']} Section: {section}. "
    if rel.get("evidence_quote"):
        desc += f"Source: '{rel['evidence_quote'].strip()}' "
    if doc:
        desc += f"Document: {doc['url']}. "
    if rel.get("version_note"):
        desc += f"Note: {rel['version_note']}"
    desc = desc.strip()
    return desc if len(desc) <= 4000 else desc[:3997] + "..."


associations, rejected = [], []
for rel in relationships:
    if REQUIRE_VERIFIED_EVIDENCE and rel["verified"] is False:
        rejected.append((rel["entry_id"], "evidence quote not found in document text"))
        continue
    version, rel["version_note"] = resolve_version(rel["entry_id"])
    if version is None and not cmr_versions(rel["entry_id"]) and not KEEP_ENTRIES_NOT_IN_CMR:
        rejected.append((rel["entry_id"], "not found in CMR"))
        continue
    umm_type = map_to_umm_c_association_type(rel["relationship"])
    assoc = {"Type": umm_type, "Description": generate_description(rel["entry_id"], umm_type, rel),
             "EntryId": rel["entry_id"]}
    if version:
        assoc["Version"] = version
    rel["association"] = assoc
    associations.append(assoc)

for entry_id, reason in rejected:
    print(f"Rejected {entry_id}: {reason}")
print(f"{len(associations)} associations built")

## Step 5 — Validate (skill Step 3)

In [ ]:
def validate_umm_c_association(association):
    validation = {"valid": True, "errors": []}

    def fail(msg):
        validation["valid"] = False
        validation["errors"].append(msg)

    if not association.get("EntryId"):
        fail("EntryId (ShortName) is required")
    if not association.get("Type"):
        fail("Type is required")
    if association.get("Type") not in VALID_ASSOCIATION_TYPES:
        fail(f"Type '{association.get('Type')}' not in valid enum: {VALID_ASSOCIATION_TYPES}")
    if "Description" in association and not 1 <= len(association["Description"]) <= 4000:
        fail(f"Description must be 1-4000 characters ({len(association['Description'])} chars)")
    if len(association.get("EntryId", "")) > 80:
        fail(f"EntryId exceeds 80 characters ({len(association['EntryId'])} chars)")
    if "Version" in association and not isinstance(association["Version"], str):
        fail("Version must be a string")
    extra = set(association) - {"Type", "Description", "EntryId", "Version"}
    if extra:
        fail(f"Fields not allowed: {sorted(extra)}")
    return validation


valid_associations = []
for a in associations:
    v = validate_umm_c_association(a)
    if v["valid"]:
        valid_associations.append(a)
    else:
        print(f"Validation error for {a.get('EntryId')}: {v['errors']}")
print(f"{len(valid_associations)}/{len(associations)} associations pass the skill's checks")

## Step 6 — Merge into the UMM-C record and validate against the official schema
The whole record is checked against the official UMM-C JSON schema for `UMM_C_VERSION`. Errors that were already in the original CMR record are reported separately, so only errors this notebook introduced count against the output.

In [ ]:
from jsonschema import Draft7Validator
from referencing import Registry
from referencing.jsonschema import DRAFT7


def load_umm_c_validator(version):
    base = f"https://cdn.earthdata.nasa.gov/umm/collection/v{version}/"
    main = session.get(base + "umm-c-json-schema.json", timeout=60).json()
    common = session.get(base + "umm-cmn-json-schema.json", timeout=60).json()
    registry = Registry().with_resource(base + "umm-cmn-json-schema.json", DRAFT7.create_resource(common))
    return Draft7Validator({**main, "$id": base + "umm-c-json-schema.json"}, registry=registry)


def schema_errors(validator, record):
    return {(".".join(map(str, e.absolute_path)) or "<root>", e.message) for e in validator.iter_errors(record)}


def merge_associations(existing, new):
    if MERGE_MODE == "replace":
        return new
    regenerated = {a["EntryId"] for a in new}
    return [a for a in existing or [] if a.get("EntryId") not in regenerated] + new


updated_umm = json.loads(json.dumps(target_umm))
merged = merge_associations(target_umm.get("MetadataAssociations"), valid_associations)
if merged:
    updated_umm["MetadataAssociations"] = merged       # schema requires minItems 1
else:
    updated_umm.pop("MetadataAssociations", None)

validator = load_umm_c_validator(UMM_C_VERSION)
pre_existing = schema_errors(validator, target_umm)
introduced = schema_errors(validator, updated_umm) - pre_existing
for path, msg in sorted(pre_existing):
    print(f"  pre-existing (original record): {path}: {msg[:200]}")
for path, msg in sorted(introduced):
    print(f"  INTRODUCED: {path}: {msg[:200]}")
print(f"UMM-C {UMM_C_VERSION} schema: {'VALID' if not introduced else 'INVALID'} "
      f"({len(introduced)} introduced, {len(pre_existing)} pre-existing errors)")

In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
stem = f"{SHORT_NAME}_V{VERSION}"
umm_path = WORK_DIR / f"{stem}_UMM_C_updated.json"
evidence_path = WORK_DIR / f"{stem}_lineage_evidence.json"

umm_path.write_text(json.dumps(updated_umm, indent=2, ensure_ascii=False) + "\n")
evidence_path.write_text(json.dumps({
    "target": {"ShortName": SHORT_NAME, "Version": VERSION, "concept_id": target_meta["concept-id"],
               "revision_id": target_meta["revision-id"]},
    "generated": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "extraction_method": extraction_method,
    "documents": [{k: d[k] for k in ("id", "label", "subtype", "url")} for d in documents],
    "relationships": relationships,
    "rejected": [{"entry_id": e, "reason": r} for e, r in rejected],
    "schema_validation": {"umm_c_version": UMM_C_VERSION, "introduced_errors": sorted(introduced),
                          "pre_existing_errors": sorted(pre_existing)},
}, indent=2, ensure_ascii=False) + "\n")

print(f"Updated UMM-C : {umm_path}")
print(f"Evidence      : {evidence_path}\n")
print(json.dumps(updated_umm.get("MetadataAssociations", []), indent=2, ensure_ascii=False))

## Optional — submit to CMR
**Review the associations before you submit.** This replaces the live collection record. It requires an Earthdata Login token with ingest permission for the provider (`EDL_TOKEN` environment variable). `validate_only=True` calls CMR's validation endpoint, which does not change the record.

In [ ]:
SUBMIT_TO_CMR = False


def submit_to_cmr(record, validate_only=True):
    provider, native_id = target_meta["provider-id"], target_meta["native-id"]
    url = (f"{CMR_BASE_URL}/ingest/providers/{provider}/validate/collection/{native_id}" if validate_only
           else f"{CMR_BASE_URL}/ingest/providers/{provider}/collections/{native_id}")
    headers = {"Content-Type": f"application/vnd.nasa.cmr.umm+json;version={UMM_C_VERSION}",
               "Accept": "application/json",
               "Authorization": f"Bearer {os.environ['EDL_TOKEN']}"}
    r = (session.post if validate_only else session.put)(url, data=json.dumps(record), headers=headers, timeout=120)
    print(r.status_code, r.text[:2000])
    return r


if SUBMIT_TO_CMR:
    submit_to_cmr(updated_umm, validate_only=True)